In [ ]:
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
%cd YOUR_REPO

In [ ]:
import gdown

file_id = "1_XdzdW8UNG1TTS6QcX10uhoS6N11OBit"
destination = "mosi_data.pkl"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}", destination, quiet=False)

In [ ]:
FILE_PATH = '/content/mosi_data.pkl'
train_data, valid_data, test_data = get_dataloader(FILE_PATH)

In [ ]:
device = get_default_device()
print(device)

In [ ]:
from src.data_loader import get_loader
from src.config import get_config

train_config = get_config("mosi", mode="train")
dev_config = get_config("mosi", mode="dev")
test_config = get_config("mosi", mode="test")

train_loader = get_loader(None, train_config)
dev_loader = get_loader(None, dev_config)
test_loader = get_loader(None, test_config)

In [ ]:
model = MultimodalEmotionModel(
    text_dim=300,
    vision_dim=35,
    audio_dim=74,
    d_model=128
)

model = model.to(device)

In [ ]:
criterion = nn.MSELoss()

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=3
)

In [ ]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0

    for batch in tqdm(loader):

        text, vision, audio, labels = batch

        text = text.to(device)
        vision = vision.to(device)
        audio = audio.to(device)
        labels = labels.to(device)

        preds = model(text, vision, audio)

        loss = criterion(preds.squeeze(), labels)

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for batch in loader:

            text, vision, audio, labels = batch

            text = text.to(device)
            vision = vision.to(device)
            audio = audio.to(device)
            labels = labels.to(device)

            preds = model(text, vision, audio)

            loss = criterion(preds.squeeze(), labels)

            total_loss += loss.item()

            preds_all.append(preds.cpu())
            labels_all.append(labels.cpu())

    return total_loss / len(loader)

In [ ]:
epochs = 20

for epoch in range(epochs):

    train_loss = train_epoch(model, train_loader)

    val_loss = evaluate(model, dev_loader)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

In [ ]:
test_loss = evaluate(model, test_loader)

print("Test Loss:", test_loss)

In [ ]:
torch.save(
    model.state_dict(),
    "multimodal_emotion_model.pth"
)

In [ ]:
model = MultimodalEmotionModel()

model.load_state_dict(
    torch.load("multimodal_emotion_model.pth")
)